# Ollama LR Extraction Tester
Use this notebook to test different Ollama models and prompts on your LR images.
Change `IMAGE_PATH` and `MODEL` in the config cell, then run all cells.

In [ ]:
# ── CONFIG — edit these two lines ──────────────────────────────────────────
IMAGE_PATH = r"C:\Users\Admin\Documents\Claude\Projects\Biling Automation Bot\data\uploads\invoices\YOUR_INVOICE_ID\YOUR_IMAGE.png"
MODEL      = "llava:7b"   # change to: llava-phi3, qwen2-vl, etc.
# ───────────────────────────────────────────────────────────────────────────

OLLAMA_URL = "http://localhost:11434"

In [ ]:
import base64, io, json, hashlib
from pathlib import Path
from PIL import Image
import httpx
from IPython.display import display
import PIL.Image

# Show the image so you can verify you loaded the right file
img = PIL.Image.open(IMAGE_PATH)
print(f"Image size: {img.size}  mode: {img.mode}  file: {Path(IMAGE_PATH).name}")
img.thumbnail((600, 600))
display(img)

In [ ]:
def prepare_image(path, max_side=1536):
    """Resize to max_side on longest edge, convert to JPEG, return base64."""
    img = Image.open(path)
    if img.mode not in ("RGB", "L"):
        img = img.convert("RGB")
    w, h = img.size
    if max(w, h) > max_side:
        scale = max_side / max(w, h)
        img = img.resize((int(w * scale), int(h * scale)), Image.LANCZOS)
    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=85)
    b64 = base64.b64encode(buf.getvalue()).decode()
    print(f"Prepared image: {img.size}  base64 size: {len(b64)} chars  hash: {hashlib.md5(b64[:500].encode()).hexdigest()[:8]}")
    return b64

image_b64 = prepare_image(IMAGE_PATH)

In [ ]:
PROMPT = """
You are reading a Mahindra Logistics Goods Consignment Note (GCN / LR).
Extract exactly the fields below. Read the image carefully.

FIELD LOCATIONS on this specific form:
- vehicle_no: top-left area, labeled 'Vehicle No.' e.g. NL01AK0496
- gcn_no: top-right area, labeled 'G.C.N.No.' — a 9-digit number e.g. 112032145.
  This is DIFFERENT from the Delivery Order No.
- gcn_date: top-right area, labeled 'Date.' next to G.C.N.No. e.g. 07-Apr-2026
- from_city: right side, small box labeled 'From' — just the city name e.g. Rajkot
- destination: right side, small box labeled 'To' — just the city name e.g. Gabhana.
  Do NOT use the full consignee address.
- qty: table column 'No of Pkg' e.g. 9 Units
- delivery_order_nos: table column labeled 'Delivery Order No' (NOT the GCN No).
  These are 10-digit numbers e.g. 7412304508. There may be multiple separated by
  commas. Return ALL as a JSON array of strings.
- delivery_date: look in the 'Proof of Delivery' box at the bottom-right,
  labeled 'Date:' — it is handwritten, e.g. 15/04/2026. Extract if readable, else null.

CRITICAL RULES:
- delivery_order_nos MUST come only from the 'Delivery Order No' table column.
  NEVER put the GCN number here.
- destination is ONLY the city from the 'To' box, not from the address block.
- Do NOT invent or guess any value not visible in the image.
- Return ONLY a JSON object, no explanation, no markdown.

{
  "vehicle_no": "<read from image>",
  "gcn_no": "<read from image>",
  "gcn_date": "<read from image>",
  "from_city": "<read from image>",
  "destination": "<read from image>",
  "qty": "<read from image>",
  "delivery_order_nos": ["<read from image>"],
  "delivery_date": "<read from image or null>"
}
"""

In [ ]:
import random

print(f"Sending to Ollama model: {MODEL} ...")

payload = {
    "model": MODEL,
    "messages": [
        {
            "role": "user",
            "content": PROMPT,
            "images": [image_b64],
        }
    ],
    "format": "json",
    "stream": False,
    "keep_alive": 0,
    "options": {
        "temperature": 0,
        "seed": random.randint(1, 999999),
    },
}

resp = httpx.post(f"{OLLAMA_URL}/api/chat", json=payload, timeout=180.0)
print(f"Status: {resp.status_code}")

raw = resp.json().get("message", {}).get("content", "")
print(f"\nRaw response ({len(raw)} chars):")
print(raw)

In [ ]:
# Parse and display the extracted fields cleanly
try:
    data = json.loads(raw)
    print("EXTRACTED FIELDS:")
    print("-" * 40)
    for k, v in data.items():
        print(f"  {k:25s}: {v}")
except json.JSONDecodeError as e:
    print(f"JSON parse error: {e}")
    print("Raw text above — model returned non-JSON")

## Try a different model
Change `MODEL` in the config cell at the top to one of these and re-run:
- `llava:7b` — default, hallucinates on forms
- `llava-phi3` — Phi-3.5 base, better instruction following, 2.9 GB
- `qwen2-vl` — best for documents, ~5 GB
- `moondream` — tiny 1.8 GB, less accurate but fast

## Find your image path
Uploaded images are saved under:
`data/uploads/invoices/<invoice_id>/<uuid>.png`

Run this to list recent uploads:

In [ ]:
import glob, os

base = r"C:\Users\Admin\Documents\Claude\Projects\Biling Automation Bot\data\uploads\invoices"
files = sorted(glob.glob(f"{base}\\**\\*.png", recursive=True) +
               glob.glob(f"{base}\\**\\*.jpg", recursive=True),
               key=os.path.getmtime, reverse=True)

print("Most recent uploaded images:")
for f in files[:10]:
    size_kb = os.path.getsize(f) // 1024
    print(f"  {size_kb:5d} KB  {f}")

In [ ]:
# Quick model check — see what models Ollama has available
r = httpx.get(f"{OLLAMA_URL}/api/tags", timeout=10)
models = r.json().get("models", [])
print("Models available in Ollama:")
for m in models:
    size_gb = m.get('size', 0) / 1e9
    print(f"  {m['name']:40s}  {size_gb:.1f} GB")